# Perspectives and visualizations

## Europe's Energy Transition: Urgency Meets Reality

Europe's energy landscape is undergoing a rapid, necessary transformation. The dual challenges of climate urgency and economic reality frame this important shift. Fully decarbonizing the continent's energy system demands immense investment, estimated at trillions of euros (European Commission, 2020), even as renewable energy technologies become increasingly cost-competitive (IRENA, 2021). Our analysis delves into how Europe is navigating these complexities, powered by insights from recent data.

### The Economic and Social Feasibility Challenge

One critical aspect is understanding if European countries can economically and socially handle this profound change. The sheer scale of historical reliance on fossil fuels is evident in the **Cumulative CO2 Emissions from Fossil Fuels in Selected European Countries (since 1950)** stacked area chart, where nations like the UK and Germany show decades of significant carbon output, followed by France, Italy, Poland, and Spain. This legacy means a substantial transition burden. This multivariate stacked are chart shows how the emissions from different European countries changed through the years.

In [24]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [51]:
import pandas as pd
import plotly.express as px

# Load the dataset
owid_co2_data_df = pd.read_csv('owid-co2-data.csv')

# Define a list of European countries
european_countries = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark',
    'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland',
    'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands',
    'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden',
    'United Kingdom', 'Norway', 'Switzerland', 'Iceland'
]

# Select a few key countries for this cumulative view (to avoid clutter)
selected_countries_cumulative = ['Germany', 'United Kingdom', 'France', 'Italy', 'Spain', 'Poland']

# Filter for selected European countries and years from 1950 onwards
df_cumulative = owid_co2_data_df[
    (owid_co2_data_df['country'].isin(selected_countries_cumulative)) &
    (owid_co2_data_df['year'] >= 1950)
].copy()

# Select the 'cumulative_co2' column and ensure it's numeric
df_cumulative_filtered = df_cumulative[['country', 'year', 'cumulative_co2']].dropna()
df_cumulative_filtered['cumulative_co2'] = pd.to_numeric(df_cumulative_filtered['cumulative_co2'])

# Contrasting colours for each country
contrast_colors = [
    '#FFD700',  # gold
    '#FF7F0E',  # orange
    '#2CA02C',  # green
    '#D62728',  # red
    '#9467BD',  # purple
    '#17BECF'   # cyan
]

# Create the stacked area chart with smooth (rounded) curves
fig = px.area(
    df_cumulative_filtered,
    x="year",
    y="cumulative_co2",
    color="country",
    line_group="country",
    hover_name="country",
    title="Cumulative CO₂ Emissions from Fossil Fuels\nin Selected European Countries (since 1950)",
    labels={
        "year": "Year",
        "cumulative_co2": "Cumulative CO₂ Emissions (million tonnes)"
    },
    color_discrete_sequence=contrast_colors,
    line_shape='spline',   # <-- smooths the corners
)

for trace in fig.data:
    trace.line.smoothing = 1.3   # of een ander getal 0–1.3


fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Cumulative CO₂ Emissions (million tonnes)",
    hovermode="x unified",
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    title_font_color='white',
    legend_font_color='white'
)

fig.update_xaxes(
    showgrid=True,
    gridcolor='rgba(255,255,255,0.2)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_yaxes(
    showgrid=False,
    tickfont_color='white',
    title_font_color='white'
)

fig.show()




This graph visually demonstrating the massive scale of Europe's historical fossil fuel reliance. It clearly shows the enormous cumulative CO2 emissions, reaching hundreds of thousands of million tonnes, proving the "sheer scale" mentioned. Crucially, the graph ranks the countries just as described: the UK and Germany have the largest emissions over decades, followed by France, Italy, Poland, and Spain. This visual evidence of unequal historical responsibility makes concrete why countries like the UK and Germany face a bigger "substantial transition burden" – they have the largest legacy of past pollution to overcome.

Financially, the journey is demanding. Our **Energy Costs per European Country (Renewable vs Fossil)** horizontal bar chart highlights the fluctuating financial commitments over time, with leading economies like France, Germany, the UK, Italy, and Spain seeing costs exceed €20 billion in 2015, peaking above €100 billion in 2022, before moderating to under €20 billion in 2025. The peak in 2022 can possibly be attributed to the increased energy demand due to the pandemic and growing tension between Russia and Ukraine (Cevik et al., 2022).

In [50]:
import pandas as pd
import plotly.express as px

# 1) Data laden en bewerken
price = pd.read_csv('european_wholesale_electricity_price_data_monthly.csv', parse_dates=['Date'])
gen = pd.read_csv('data2.csv', skiprows=7,
                  names=['Country','Time','Balance','Product','Value','Unit'],
                  low_memory=False)

gen = gen[gen['Balance']=='Net Electricity Production'].copy()
gen['Time'] = pd.to_datetime(gen['Time'], format='%B %Y')
gen['Value'] = pd.to_numeric(gen['Value'], errors='coerce')

total_prod = gen[gen['Product']=='Electricity'][['Country','Time','Value']].rename(columns={'Value':'Total_GWh'})
renew_prod = gen[gen['Product']=='Total Renewables (Hydro, Geo, Solar, Wind, Other)'][['Country','Time','Value']].rename(columns={'Value':'Renewable_GWh'})

df = total_prod.merge(renew_prod, on=['Country','Time'], how='inner')
df = df.merge(
    price[['Country','Date','Price (EUR/MWhe)']].rename(columns={'Date':'Time','Price (EUR/MWhe)':'Price'}),
    on=['Country','Time'], how='inner'
)

df['Total_MWh']     = df['Total_GWh'] * 1000
df['Renewable_MWh'] = df['Renewable_GWh'] * 1000
df['Cost_Total']    = df['Price'] * df['Total_MWh']
df['Cost_Renewable']= df['Price'] * df['Renewable_MWh']
df['Cost_Fossil']   = df['Cost_Total'] - df['Cost_Renewable']

df['Year'] = df['Time'].dt.year
agg = df.groupby(['Year','Country'])[['Cost_Renewable','Cost_Fossil','Cost_Total']].sum() / 1e9
agg = agg.reset_index()

# 2) Fixed vertical order gebaseerd op laatste jaar
last_year   = agg['Year'].max()
fixed_order = (
    agg[agg['Year']==last_year]
    .sort_values('Cost_Total', ascending=False)['Country']
    .tolist()
)

# 3) Maak animated bar chart met fixed order, Renewable in groen
fig = px.bar(
    agg,
    x=['Cost_Renewable','Cost_Fossil'],
    y='Country',
    orientation='h',
    animation_frame='Year',
    category_orders={'Year': sorted(agg['Year'].unique()), 'Country': fixed_order},
    labels={'value':'Cost (billion EUR)','variable':'Type','Country':'Country','Year':'Year'},
    title='Energy Costs per European Country (Renewable vs Fossil)',
    color_discrete_map={'Cost_Renewable':'#2CA02C'}  # Renewable in groen
)

fig.update_layout(
    barmode='stack',
    height=800,
    margin=dict(l=50, r=20, t=50, b=50),
    transition={'duration':1000,'easing':'cubic-in-out'}
)

# 4) Smooth slider
if fig.layout.sliders:
    fig.layout.sliders[0].transition = {'duration':1000,'easing':'cubic-in-out'}

# 5) Fixed reversed y-axis
fig.update_yaxes(autorange='reversed', categoryorder='array', categoryarray=fixed_order)

# 6) Definieer per-jaar x-as limieten
year_max = {
    **{yr: 30  for yr in range(2015, 2021)},
    **{yr: 130 for yr in (2021,2022,2023)},
    2025: 30
}

# 7) Pas per-frame x-as en forceer redraw
for frame in fig.frames:
    yr = int(frame.name)
    if yr in year_max:
        frame.layout.xaxis.range     = [0, year_max[yr]]
        frame.layout.xaxis.autorange = False
    else:
        frame.layout.xaxis.autorange = True
    frame.layout.yaxis = {'autorange':'reversed'}

updater = fig.layout.updatemenus[0].buttons[0].args[1]
updater['frame']['redraw']      = True
updater['transition']['duration']= 1000
updater['transition']['easing']  = 'cubic-in-out'
updater['frame']['duration']     = 2000
updater['frame']['easing']       = 'cubic-in-out'

# 8) Optioneel: forceer basislayout op langst geldende max
global_max = max(year_max.values())
fig.update_xaxes(range=[0, global_max], autorange=False, rangemode='tozero')

fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084'
)

# As- en gridlijnen wit
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.2)', zerolinecolor='white', color='white')
fig.update_yaxes(showgrid=False, color='white')

# Titel- en legend-teksten wit
fig.update_layout(
    title_font_color='white',
    legend_font_color='white'
)

# Slider en knoppen styling
if fig.layout.sliders:
    s = fig.layout.sliders[0]
    s.bgcolor      = '#004084'
    s.bordercolor  = 'white'
    s.borderwidth  = 1
    s.font.color   = 'white'
    s.currentvalue.font.color = 'white'

if fig.layout.updatemenus:
    u = fig.layout.updatemenus[0]
    u.bgcolor     = '#004084'
    u.bordercolor = 'white'
    u.borderwidth = 1
    u.font.color  = 'white'

# 9) Render
fig.show()



This volatility in investment is further reflected in the **Wholesale Electricity Price Trends in Select European Countries** line chart, which shows prices largely below €100/MWh until a dramatic surge to over €500/MWh in mid-2022 for some countries, before returning to more moderate levels by 2023. Such price swings underscore the inherent market risks and the need for robust financial strategies during the transition. This chart shows the same peak in 2022 the previous chart shows. This can also be attributed to the increased energy demand due to the pandemic and growing tension between Russia and Ukraine (Cevik et al., 2022).

In [71]:
import pandas as pd
import plotly.express as px

# 1) Data inladen en filteren
df = pd.read_csv('european_wholesale_electricity_price_data_monthly.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df[df['Date'].dt.year >= 2015]
countries = ['Germany', 'France', 'United Kingdom', 'Spain', 'Italy', 'Netherlands']
df = df[df['Country'].isin(countries)]

# 2) Zeer contrastrijke kleuren (geen blauwtint)
contrast_colors = ['#FF0000','#FFA500','#FFFF00','#00FF00','#00FFFF','#FF00FF']

# 3) Maak interactieve line chart met extra grote afmetingen
fig = px.line(
    df,
    x='Date',
    y='Price (EUR/MWhe)',
    color='Country',
    title='Wholesale Electricity Price Trends in Selected European Countries Deze is dubbel',
    color_discrete_sequence=contrast_colors,
    labels={'Price (EUR/MWhe)':'Price (EUR/MWhe)', 'Date':'Date'},
    width=1600,     # <<< breder
    height=900      # <<< hoger
)

# 4) Achtergrond en styling
fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    legend_title_font_color='white'
)

# 5) Assen en grid in wit
fig.update_xaxes(
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_yaxes(
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)

fig.show()






**Note:** See prices jump from stable (<€100) to extreme (>€500) in 2022, proving the volatility described above and the need for strong financial planning during the energy transition.

The ability to invest varies significantly across the continent. The **Correlation between GDP and CO2 emissions for EU Countries (2022)** scatterplot with a trendline clearly illustrates that wealthier countries such as Germany, Spain, Italy, and France tend to have higher overall CO2 emissions, reflecting a historical link between economic development and carbon intensity (Apeti et al., 2025). Conversely, countries with lower GDPs, like Austria, Greece, Slovakia, and Estonia, generally show lower emissions.

In [77]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Data inladen en filteren
url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
df = pd.read_csv(url)
df = df[df['year'] == 2022]
eu_countries = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia',
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
    'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
    'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania',
    'Slovakia', 'Slovenia', 'Spain', 'Sweden'
]
df = df[df['country'].isin(eu_countries)].dropna(subset=['gdp', 'co2'])

# 2) Log-transformatie
df['log_gdp'] = np.log10(df['gdp'])
df['log_co2'] = np.log10(df['co2'])

# 3) Maak interactieve scatter met regressielijn
fig = px.scatter(
    df,
    x='log_gdp',
    y='log_co2',
    text='country',
    trendline='ols',
    trendline_color_override='white',
    title='Correlation between GDP and CO₂ emissions for EU countries (2022) – log-scale',
    labels={'log_gdp':'Log10 GDP (dollars)', 'log_co2':'Log10 CO₂ (ton)'},
    width=1600,
    height=1000
)

# 4) Marker-punten wít maken
fig.update_traces(
    marker=dict(color='white', size=12, line=dict(color='white', width=1)),
    selector=lambda trace: 'markers' in trace.mode
)

# 5) Landnamen in geel en nét boven de stip zetten
fig.update_traces(
    textfont=dict(color="#FFFFFF", size=16),
    textposition='top center',
    selector=lambda trace: 'text' in trace.mode
)

# 6) Styling: donkerblauwe ondergrond en witte assen/grid/tekst
fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24
)

fig.update_xaxes(
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_yaxes(
    showgrid=True,
    gridcolor='rgba(255,255,255,0.3)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)

fig.show()






 This graph highlights the complex challenge of decoupling economic growth from emissions while also suggesting that wealthier nations may be better equipped to absorb the significant costs of decarbonization.

Despite the financial weight, many European nations are actively pushing forward. The **Renewable electricity share in European Production (March 2025)** chloropleth map showcases considerable progress in countries like Iceland and Norway, nearing 100% renewable energy usage. Countries such as Croatia, Portugal, and Luxembourg also demonstrate strong renewable shares, around 80%. However, major economies like France, Italy, the Netherlands, and Germany remain under 50%, indicating that substantial work is still required to integrate renewables fully. These disparities reflect a combination of natural resource availability, existing infrastructure, policy environments, and financial capacity.

In [66]:
import pandas as pd
import plotly.express as px

# --- Data inladen en bewerken ---
owid = pd.read_csv('owid-co2-data.csv')
prod = pd.read_csv('data2.csv', skiprows=8)

european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]

# Filter en periode
prod = prod[(prod['Balance']=='Net Electricity Production') &
            (prod['Unit']=='GWh') &
            (prod['Country'].isin(european_countries))].copy()
prod['Date_Parsed'] = pd.to_datetime(prod['Time'], format='%B %Y', errors='coerce')
latest = prod['Date_Parsed'].max()
prod = prod[prod['Date_Parsed']==latest]

# Opbouwen shares
renew = prod[prod['Product'].isin(['Hydro','Solar','Wind','Biofuels','Geothermal','Other renewables'])]
tot   = prod[prod['Product']=='Electricity']
df_ren = renew.groupby('Country')['Value'].sum().rename('Renewable_GWh')
df_tot = tot  .groupby('Country')['Value'].sum().rename('Total_GWh')
df = pd.concat([df_ren, df_tot], axis=1).dropna()
df['Share'] = df['Renewable_GWh']/df['Total_GWh']*100
df = df.reset_index()

# ISO codes join
iso = owid[['country','iso_code']].dropna().drop_duplicates()
df = df.merge(iso, left_on='Country', right_on='country', how='left').dropna(subset=['iso_code'])

# --- Verticale bar chart ---
df_sorted = df.sort_values('Share', ascending=False)  # grootste eerst

fig = px.bar(
    df_sorted,
    x='Country',
    y='Share',
    color='Share',
    color_continuous_scale='Greens',
    title=f'Renewable Electricity Share by Country – {latest.strftime("%B %Y")}',
    labels={'Share':'Renewable Share (%)','Country':''},
    width=1000,
    height=800
)

# Styling
fig.update_layout(
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    title_font_size=24,
    margin=dict(l=80, r=50, t=80, b=150)
)
fig.update_yaxes(
    range=[0,100],
    gridcolor='rgba(255,255,255,0.2)',
    zerolinecolor='white',
    tickfont_color='white',
    title_font_color='white'
)
fig.update_xaxes(
    tickfont_color='white',
    tickangle=45
)

fig.show()




The immediate economic landscape is also shaped by electricity prices. The **Wholesale Electricity Price in Europe (June 2025)** chloropleth map shows a striking range, with prices lowest in regions like France, Finland, and Sweden (under €30/MWh), likely due to their high nuclear or hydro capacity. In contrast, countries like Italy, Ireland, and Poland face significantly higher prices (above €80/MWh, with Italy nearing €110/MWh). These price variations can impact the competitiveness of businesses, household energy bills, and ultimately, the social acceptance and pace of the energy transition.


In [99]:
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go  # toegevoegd voor extra trace

# Data inladen en filteren
df = pd.read_csv('european_wholesale_electricity_price_data_monthly.csv', parse_dates=['Date'])
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df = df[df['Country'].isin(european_countries)].dropna(subset=['Price (EUR/MWhe)'])

# Basis line chart met per-land lijnen
fig = px.line(
    df,
    x='Date',
    y='Price (EUR/MWhe)',
    color='Country',
    title='Electricity Price Trends in Selected European Countries',
    labels={'Price (EUR/MWhe)':'Price (EUR/MWhe)','Date':'Date'},
    width=1300, height=850
)

# 1) Voeg Europese trendlijn (gemiddelde) toe in zwart
avg_df = df.groupby('Date')['Price (EUR/MWhe)'].mean().reset_index()
fig.add_trace(
    go.Scatter(
        x=avg_df['Date'],
        y=avg_df['Price (EUR/MWhe)'],
        mode='lines',
        line=dict(color='black', width=4),
        name='Europe Average'
    )
)

# 2) Zet Europe Average bovenaan in de legend
all_traces   = list(fig.data)
avg_trace    = [t for t in all_traces if t.name == 'Europe Average'][0]
other_traces = [t for t in all_traces if t.name != 'Europe Average']
fig.data      = tuple([avg_trace] + other_traces)

# 3) Definieer regio’s voor dropdown-filter
west  = ['Belgium','France','Ireland','Luxembourg','Netherlands','United Kingdom']
south = ['Croatia','Cyprus','Greece','Italy','Malta','Portugal','Spain']
north = ['Denmark','Estonia','Finland','Iceland','Latvia','Lithuania','Norway','Sweden']
east  = ['Bulgaria','Czechia','Hungary','Poland','Romania','Slovakia','Slovenia']

# 4) Maak lijst met alle trace-namen (inclusief Europe Average)
trace_names = [t.name for t in fig.data]

# 5) Buttons inclusief “Europe Average Only”
buttons = [
    dict(label='All Countries',
         method='update',
         args=[{'visible': [True]*len(trace_names)}, {}]),
    dict(label='Europe Average Only',
         method='update',
         args=[{'visible': [name=='Europe Average' for name in trace_names]},
               {'title':'Europe Average Only'}]),
    dict(label='Western Europe',
         method='update',
         args=[{'visible': [name in west or name=='Europe Average' for name in trace_names]},
               {'title':'Western Europe'}]),
    dict(label='Southern Europe',
         method='update',
         args=[{'visible': [name in south or name=='Europe Average' for name in trace_names]},
               {'title':'Southern Europe'}]),
    dict(label='Northern Europe',
         method='update',
         args=[{'visible': [name in north or name=='Europe Average' for name in trace_names]},
               {'title':'Northern Europe'}]),
    dict(label='Eastern Europe',
         method='update',
         args=[{'visible': [name in east  or name=='Europe Average' for name in trace_names]},
               {'title':'Eastern Europe'}])
]

# 6) Voeg dropdown en legend-click functionaliteit toe, en marge onder text
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15, y=1.15,
        xanchor='right', yanchor='top'
    )],
    legend=dict(
        title='Country',
        itemclick='toggle',
        itemdoubleclick='toggleothers'
    ),
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
    margin=dict(l=50, r=50, t=80, b=200)  # increased bottom margin
)

# 7) Styling assen en grid
fig.update_xaxes(
    showgrid=True, gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white', title_font_color='white'
)
fig.update_yaxes(
    showgrid=True, gridcolor='rgba(255,255,255,0.2)',
    tickfont_color='white', title_font_color='white'
)

# 8) Onder de grafiek instructietekst in het Engels toevoegen, lager geplaatst
fig.update_layout(
    annotations=[dict(
        text="Click a country in the legend to toggle individual lines.<br>"
             "Use the dropdown at top right to quickly filter Western, Eastern, Northern, or Southern Europe.",
        xref='paper', yref='paper',
        x=0, y=-0.18,   # moved down for clear separation
        showarrow=False,
        font=dict(color='white', size=12),
        align='left'
    )]
)

fig.show()











### The Climate Urgency Imperative

Beyond economic considerations, the climate crisis itself demands an accelerated energy transition. The urgency is underscored by Europe's current carbon footprint. The **Total CO2 emissions in Europe (2023 data)** chloropleth map illustrates that Germany, France, Poland, and the United Kingdom are still among the highest emitters, with Germany alone approaching 600 million tonnes. These current emission levels contribute directly to global warming and its pervasive impacts.

In [76]:
import pandas as pd
import plotly.express as px

# Load the dataset
owid_co2_data_df = pd.read_csv('owid-co2-data.csv')

# Define a list of European countries
european_countries = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark',
    'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland',
    'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands',
    'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden',
    'United Kingdom', 'Norway', 'Switzerland', 'Iceland'
]

# Filter for European countries and select relevant columns
df_europe_co2 = owid_co2_data_df[owid_co2_data_df['country'].isin(european_countries)].copy()

# Find the latest year with CO2 data for each country
latest_year_co2 = df_europe_co2.dropna(subset=['co2']).groupby('country')['year'].max().reset_index()
df_latest_co2_data = pd.merge(df_europe_co2, latest_year_co2, on=['country', 'year'])

# Ensure 'iso_code' is available for plotting (Plotly needs ISO codes for countries)
df_latest_co2_data.dropna(subset=['iso_code', 'co2'], inplace=True)

# Create the choropleth map
fig = px.choropleth(df_latest_co2_data,
                    locations="iso_code",
                    color="co2", # Data to be colored
                    hover_name="country",
                    hover_data={"co2": ":.2f", "year": True}, # Show CO2 formatted and year
                    color_continuous_scale=px.colors.sequential.Plasma, # Choose a color scale
                    projection="natural earth", # A good projection for global/continental maps
                    title=f"Total CO2 Emissions in Europe ({df_latest_co2_data['year'].max()} Data dubbel)",
                    labels={'co2': 'Total CO2 Emissions (million tonnes)'})

# Focus the map on Europe (adjusting the scope and center)
fig.update_geos(
    scope='europe',
    lataxis_range=[30, 75], # Latitude range for Europe
    lonaxis_range=[-25, 45], # Longitude range for Europe
    showcountries=True, countrycolor="Black"
)

fig.update_layout(height=600) # Adjust height for better visibility

# Show the plot
fig.show()


**Note:** Seeing these massive emissions next to other countries makes it clear why they demand urgent cuts – their pollution directly drives the climate crisis, forcing an accelerated switch to clean energy.

However, many European countries have made substantial progress in reducing their emissions over time. The **Percentage Change in CO2 Emissions (2000 - 2023) in European Countries** map, presented as a subdivided rectangle, illustrates this varied success. Nations like Denmark, Portugal, the Netherlands, Ireland, and Austria have achieved significant reductions (e.g., Denmark -49.78%, Portugal -43.30%). However, a few countries, notably Iceland (+28.26%) and Lithuania (+4.92%), have seen increases, highlighting that the decarbonization pathway is not uniform and requires continued, tailored efforts.

In [64]:
import pandas as pd
import plotly.graph_objects as go

# 1) Data inladen en filteren
df = pd.read_csv('owid-co2-data.csv')
european_countries = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark',
    'Estonia','Finland','France','Germany','Greece','Hungary','Ireland',
    'Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden',
    'United Kingdom','Norway','Switzerland','Iceland'
]
df = df[df['country'].isin(european_countries)].dropna(subset=['co2'])

# 2) Filter op jaren (bijv. vanaf 2000)
start_year = 2000
years = sorted(df['year'].unique())
years = [y for y in years if y >= start_year]
df = df[df['year'].isin(years)]

# 3) Pivot voor heatmap: landen × jaren
heatmap_df = df.pivot(index='country', columns='year', values='co2')
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.index))  # optioneel: alfabetisch

# 4) Bouw Figure met clear cell boundaries
fig = go.Figure(data=go.Heatmap(
    z=heatmap_df.values,
    x=heatmap_df.columns,
    y=heatmap_df.index,
    colorscale='RdYlGn_r',
    colorbar=dict(title='CO₂ Emissions (Mt)'),
    hovertemplate="Country: %{y}<br>Year: %{x}<br>CO₂: %{z:.2f} Mt<extra></extra>"
))

# 5) Vakjes duidelijk maken
fig.update_traces(xgap=1, ygap=1)

# 6) Axis ticks voor elk jaar en land
fig.update_xaxes(
    tickmode='array',
    tickvals=heatmap_df.columns,
    ticktext=heatmap_df.columns,
    tickangle=45,
    title_text='Year',
    tickfont_color='white'
)
fig.update_yaxes(
    tickmode='array',
    tickvals=heatmap_df.index,
    ticktext=heatmap_df.index,
    title_text='Country',
    tickfont_color='white'
)

# 7) Layout & styling (background blue, title in English)
fig.update_layout(
    title='Annual CO₂ Emissions per European Country (2000–' + str(years[-1]) + ')',
    width=1200, height=900,
    margin=dict(l=150, r=50, t=80, b=150),
    paper_bgcolor='#004084',
    plot_bgcolor='#004084',
    font_color='white',
)

fig.show()





Climate scientists consistently warn of limited time to act effectively, with global CO2 emissions needing drastic cuts by 2030 to limit warming to 1.5°C (IPCC, 2022). The ongoing presence of high total emissions in key European economies, as shown by the **Total CO2 emissions in Europe (2023 data)** map, underscores the immediate imperative for these nations to accelerate their transition.


In conclusion, Europe's energy transition is a complex balancing act between climate imperative and economic reality. While some countries are leading the way in renewable adoption and emission reductions, the continent still faces significant challenges in decarbonizing major economies, managing price volatility, and addressing the legacy of fossil fuel use. The insights from these eight graphs collectively paint a detailed picture of progress, persistent hurdles, and the immense, urgent task ahead for Europe to secure a clean and reliable energy future.